# Banco Inter Document Import Testing

This notebook demonstrates and tests the new Banco Inter document import functionality.
The system supports importing four types of financial documents from Banco Inter:

1. **Relatório Mensal de Investimentos** (Monthly Investment Reports)
2. **Nota de Corretagem** (Brokerage Notes)
3. **Extrato** (Bank Statements)
4. **Relatório Consolidado** (Consolidated Reports) - NEW with PDF support

## Features Tested
- File format validation
- Brazilian number format parsing
- Portuguese date parsing
- PDF processing for consolidated reports
- API endpoints for document upload and management
- Asset creation and portfolio management


## Setup and Imports

In [ ]:
import os
import sys
import django
import requests
import json
import tempfile
import pandas as pd
from pathlib import Path
from decimal import Decimal
from datetime import datetime, date
from io import StringIO

# Add the project root to Python path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

# Setup Django
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings.local')
django.setup()

print(f"Project root: {project_root}")
print("Django setup complete")

In [ ]:
# Import Django models and services
from django.contrib.auth import get_user_model
from personal_finance.data_sources.models import DocumentImport
from personal_finance.data_sources.importers import (
    BancoInterImportService,
    BancoInterMonthlyReportParser,
    BancoInterConsolidatedReportParser,
    BancoInterBrokerageNoteParser,
    BancoInterExtractParser,
    PDF_AVAILABLE
)
from personal_finance.assets.models import Asset
from personal_finance.portfolios.models import Portfolio, Position, Transaction

User = get_user_model()

print(f"PDF processing available: {PDF_AVAILABLE}")
print(f"Available document types: {[choice[0] for choice in DocumentImport.DOCUMENT_TYPES]}")

## Test User Setup

In [ ]:
# Create or get a test user
test_user, created = User.objects.get_or_create(
    username='banco_inter_test_user',
    defaults={
        'email': 'test@bancointer.com',
        'first_name': 'Banco Inter',
        'last_name': 'Test User'
    }
)

print(f"Test user: {test_user.username} ({'created' if created else 'existing'})")
print(f"User ID: {test_user.id}")

## 1. Test Brazilian Number Format Parsing

Testing the parsing of Brazilian number formats commonly found in Banco Inter documents.

In [ ]:
# Test decimal parsing with Brazilian formats
def test_decimal_parsing():
    """Test Brazilian number format parsing"""
    
    # Create a temporary parser instance for testing
    with tempfile.NamedTemporaryFile(suffix='.csv') as f:
        parser = BancoInterMonthlyReportParser(f.name, test_user)
        
        test_cases = [
            ('R$ 1.234,56', Decimal('1234.56')),
            ('1.234,56', Decimal('1234.56')),
            ('(123,45)', Decimal('-123.45')),
            ('R$ 10.000,00', Decimal('10000.00')),
            ('2,50', Decimal('2.50')),
            ('15.678,90', Decimal('15678.90')),
            ('(R$ 500,75)', Decimal('-500.75')),
        ]
        
        results = []
        for input_str, expected in test_cases:
            result = parser._parse_decimal(input_str)
            success = result == expected
            results.append({
                'input': input_str,
                'expected': str(expected),
                'result': str(result),
                'success': success
            })
            
        return pd.DataFrame(results)

decimal_test_results = test_decimal_parsing()
print("Brazilian Number Format Parsing Test Results:")
print(decimal_test_results.to_string(index=False))
print(f"\nSuccess rate: {decimal_test_results['success'].mean():.1%}")

## 2. Test Date Parsing

Testing both standard and Portuguese date formats used in Banco Inter documents.

In [ ]:
def test_date_parsing():
    """Test date parsing including Portuguese formats"""
    
    with tempfile.NamedTemporaryFile(suffix='.csv') as f:
        standard_parser = BancoInterMonthlyReportParser(f.name, test_user)
        
    # Test standard date formats
    standard_test_cases = [
        ('31/12/2024', datetime(2024, 12, 31)),
        ('01/01/2025', datetime(2025, 1, 1)),
        ('15-08-2024', datetime(2024, 8, 15)),
        ('2024-12-25', datetime(2024, 12, 25)),
        ('29.02.2024', datetime(2024, 2, 29)),  # Leap year
    ]
    
    results = []
    
    # Test standard formats
    for input_str, expected in standard_test_cases:
        result = standard_parser._parse_date(input_str)
        success = result == expected if result else False
        results.append({
            'type': 'Standard',
            'input': input_str,
            'expected': expected.strftime('%Y-%m-%d') if expected else 'None',
            'result': result.strftime('%Y-%m-%d') if result else 'None',
            'success': success
        })
    
    # Test Portuguese date formats (for consolidated reports)
    if PDF_AVAILABLE:
        with tempfile.NamedTemporaryFile(suffix='.pdf') as f:
            pdf_parser = BancoInterConsolidatedReportParser(f.name, test_user)
            
            portuguese_test_cases = [
                ('29 de Agosto de 2025', datetime(2025, 8, 29)),
                ('15 de Janeiro de 2024', datetime(2024, 1, 15)),
                ('31 de Dezembro de 2023', datetime(2023, 12, 31)),
                ('1 de Maio de 2024', datetime(2024, 5, 1)),
            ]
            
            for input_str, expected in portuguese_test_cases:
                result = pdf_parser._extract_date_from_line(input_str)
                success = result == expected if result else False
                results.append({
                    'type': 'Portuguese',
                    'input': input_str,
                    'expected': expected.strftime('%Y-%m-%d') if expected else 'None',
                    'result': result.strftime('%Y-%m-%d') if result else 'None',
                    'success': success
                })
    
    return pd.DataFrame(results)

date_test_results = test_date_parsing()
print("Date Parsing Test Results:")
print(date_test_results.to_string(index=False))
print(f"\nOverall success rate: {date_test_results['success'].mean():.1%}")
if PDF_AVAILABLE:
    portuguese_success = date_test_results[date_test_results['type'] == 'Portuguese']['success'].mean()
    print(f"Portuguese date parsing success rate: {portuguese_success:.1%}")

## 3. Test Sample CSV File Creation and Import

Creating sample CSV files for different document types and testing the import process.

In [ ]:
def create_sample_monthly_report():
    """Create a sample monthly investment report CSV"""
    data = """
Ativo,Posição,Valor Atual,Rentabilidade
PETR4,100,"R$ 2.750,00","5,25%"
VALE3,200,"R$ 6.840,00","3,15%"
ITUB4,150,"R$ 4.320,00","7,80%"
BBAS3,80,"R$ 3.200,00","2,90%"
ABEV3,300,"R$ 4.500,00","1,75%"
""".strip()
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False, encoding='utf-8') as f:
        f.write(data)
        return f.name

def create_sample_brokerage_note():
    """Create a sample brokerage note CSV"""
    data = """
Data,Papel,Tipo,Quantidade,Preço,Taxa
01/08/2025,PETR4,Compra,50,"R$ 27,50","R$ 5,00"
02/08/2025,VALE3,Compra,100,"R$ 34,20","R$ 8,50"
03/08/2025,ITUB4,Venda,25,"R$ 28,80","R$ 3,20"
05/08/2025,BBAS3,Compra,40,"R$ 40,00","R$ 6,80"
""".strip()
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False, encoding='utf-8') as f:
        f.write(data)
        return f.name

def create_sample_extract():
    """Create a sample bank extract CSV"""
    data = """
Data,Descrição,Valor,Saldo
01/08/2025,"Transferência recebida","R$ 5.000,00","R$ 15.000,00"
02/08/2025,"Aplicação CDB","(R$ 3.000,00)","R$ 12.000,00"
03/08/2025,"Juros recebidos","R$ 125,50","R$ 12.125,50"
05/08/2025,"Taxa de manutenção","(R$ 15,00)","R$ 12.110,50"
08/08/2025,"Resgate CDB","R$ 1.500,00","R$ 13.610,50"
""".strip()
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False, encoding='utf-8') as f:
        f.write(data)
        return f.name

# Create sample files
monthly_report_file = create_sample_monthly_report()
brokerage_note_file = create_sample_brokerage_note()
extract_file = create_sample_extract()

print("Sample files created:")
print(f"Monthly Report: {monthly_report_file}")
print(f"Brokerage Note: {brokerage_note_file}")
print(f"Extract: {extract_file}")

# Display sample content
print("\nSample Monthly Report Content:")
with open(monthly_report_file, 'r', encoding='utf-8') as f:
    print(f.read())

## 4. Test Document Import Service

Testing the complete import workflow using the BancoInterImportService.

In [ ]:
def test_document_import(file_path, document_type, expected_count=None):
    """Test importing a document using the import service"""
    
    import_service = BancoInterImportService()
    
    try:
        # Import the document
        import_record = import_service.import_document(
            file_path=file_path,
            document_type=document_type,
            user=test_user
        )
        
        return {
            'success': True,
            'import_id': import_record.id,
            'status': import_record.status,
            'imported_count': import_record.imported_transactions_count,
            'error_message': import_record.error_message,
            'filename': import_record.original_filename
        }
    
    except Exception as e:
        return {
            'success': False,
            'error': str(e),
            'import_id': None,
            'status': 'FAILED',
            'imported_count': 0
        }

# Test importing the sample files
import_tests = [
    (monthly_report_file, 'BANCO_INTER_MONTHLY_REPORT', 'Monthly Report'),
    (brokerage_note_file, 'BANCO_INTER_BROKERAGE_NOTE', 'Brokerage Note'),
    (extract_file, 'BANCO_INTER_EXTRACT', 'Extract'),
]

import_results = []

for file_path, doc_type, description in import_tests:
    print(f"\nTesting {description} import...")
    result = test_document_import(file_path, doc_type)
    result['document_type'] = description
    import_results.append(result)
    
    if result['success']:
        print(f"✅ SUCCESS: Imported {result['imported_count']} items (Import ID: {result['import_id']})")
    else:
        print(f"❌ FAILED: {result['error']}")

# Summary table
results_df = pd.DataFrame(import_results)
print("\n" + "="*60)
print("IMPORT TEST SUMMARY")
print("="*60)
print(results_df[['document_type', 'success', 'status', 'imported_count']].to_string(index=False))

success_rate = results_df['success'].mean()
print(f"\nOverall import success rate: {success_rate:.1%}")

## 5. Test Consolidated Report PDF Parsing

Testing the new PDF parsing functionality with the real Banco Inter consolidated report.

In [ ]:
def test_consolidated_report_parsing():
    """Test parsing the actual consolidated report PDF"""
    
    if not PDF_AVAILABLE:
        return "PDF processing not available - pdfplumber not installed"
    
    # Path to the sample consolidated report
    sample_pdf_path = project_root / "personal_finance" / "data_sources" / "tests" / "sample_files" / "relatorio-2025-08-31_password_removed.pdf"
    
    if not sample_pdf_path.exists():
        return f"Sample PDF not found at {sample_pdf_path}"
    
    try:
        # Test format validation
        parser = BancoInterConsolidatedReportParser(str(sample_pdf_path), test_user)
        
        print("Testing PDF format validation...")
        is_valid = parser.validate_format()
        print(f"Format validation result: {'✅ PASSED' if is_valid else '❌ FAILED'}")
        
        if not is_valid:
            return "PDF format validation failed"
        
        print("\nTesting PDF parsing...")
        parsed_data = parser.parse()
        
        # Analyze parsed data
        positions = parsed_data.get('positions', [])
        transactions = parsed_data.get('transactions', [])
        
        print(f"\n📊 PARSING RESULTS:")
        print(f"   Positions found: {len(positions)}")
        print(f"   Transactions found: {len(transactions)}")
        print(f"   Report date: {parsed_data.get('report_date')}")
        print(f"   Source: {parsed_data.get('source')}")
        
        # Show sample positions
        if positions:
            print("\n💼 SAMPLE POSITIONS (first 5):")
            for i, pos in enumerate(positions[:5]):
                print(f"   {i+1}. {pos['symbol']}: R$ {pos.get('current_balance', 0):,.2f}")
        
        # Show sample transactions
        if transactions:
            print("\n💰 SAMPLE TRANSACTIONS (first 5):")
            for i, trans in enumerate(transactions[:5]):
                amount = trans.get('amount', 0)
                desc = trans.get('description', 'N/A')[:50] + '...' if len(trans.get('description', '')) > 50 else trans.get('description', 'N/A')
                print(f"   {i+1}. R$ {amount:,.2f} - {desc}")
        
        # Test actual import
        print("\n🔄 Testing full import process...")
        import_result = test_document_import(str(sample_pdf_path), 'BANCO_INTER_CONSOLIDATED_REPORT')
        
        if import_result['success']:
            print(f"✅ IMPORT SUCCESS: {import_result['imported_count']} items imported")
        else:
            print(f"❌ IMPORT FAILED: {import_result['error']}")
        
        return {
            'validation_passed': is_valid,
            'positions_count': len(positions),
            'transactions_count': len(transactions),
            'import_success': import_result['success'],
            'imported_count': import_result.get('imported_count', 0)
        }
        
    except Exception as e:
        error_msg = f"Error testing consolidated report: {e}"
        print(f"❌ {error_msg}")
        return error_msg

# Run the consolidated report test
print("TESTING CONSOLIDATED REPORT PDF PARSING")
print("="*50)
consolidated_result = test_consolidated_report_parsing()
print("\nConsolidated report test completed.")

## 6. Check Created Assets and Portfolio Data

Examining the assets, portfolios, and transactions created by the import process.

In [ ]:
def analyze_imported_data():
    """Analyze the data created by the import process"""
    
    # Get the Banco Inter portfolio
    try:
        banco_inter_portfolio = Portfolio.objects.get(user=test_user, name="Banco Inter Import")
        print(f"📁 Portfolio: {banco_inter_portfolio.name}")
        print(f"   Description: {banco_inter_portfolio.description}")
        print(f"   Active: {banco_inter_portfolio.is_active}")
        print(f"   Created: {banco_inter_portfolio.created}")
    except Portfolio.DoesNotExist:
        print("❌ Banco Inter Import portfolio not found")
        return
    
    # Get assets created
    assets = Asset.objects.filter(currency='BRL', exchange='B3')
    print(f"\n🏭 ASSETS CREATED: {assets.count()} total")
    
    if assets.exists():
        assets_df = pd.DataFrame([
            {
                'symbol': asset.symbol,
                'name': asset.name,
                'type': asset.asset_type,
                'currency': asset.currency,
                'exchange': asset.exchange
            }
            for asset in assets[:10]  # Show first 10
        ])
        print(assets_df.to_string(index=False))
        if assets.count() > 10:
            print(f"... and {assets.count() - 10} more")
    
    # Get positions
    positions = Position.objects.filter(portfolio=banco_inter_portfolio)
    print(f"\n💼 POSITIONS: {positions.count()} total")
    
    if positions.exists():
        positions_data = []
        for pos in positions[:10]:  # Show first 10
            positions_data.append({
                'asset': pos.asset.symbol,
                'quantity': float(pos.quantity),
                'avg_cost': float(pos.average_cost),
                'first_purchase': pos.first_purchase_date
            })
        
        positions_df = pd.DataFrame(positions_data)
        print(positions_df.to_string(index=False))
        if positions.count() > 10:
            print(f"... and {positions.count() - 10} more")
    
    # Get transactions
    all_transactions = Transaction.objects.filter(
        position__portfolio=banco_inter_portfolio
    ).order_by('-transaction_date')
    
    print(f"\n💰 TRANSACTIONS: {all_transactions.count()} total")
    
    if all_transactions.exists():
        transactions_data = []
        for trans in all_transactions[:10]:  # Show first 10
            transactions_data.append({
                'date': trans.transaction_date,
                'asset': trans.position.asset.symbol,
                'type': trans.transaction_type,
                'quantity': float(trans.quantity),
                'price': float(trans.price),
                'fees': float(trans.fees),
                'notes': trans.notes[:50] + '...' if len(trans.notes) > 50 else trans.notes
            })
        
        transactions_df = pd.DataFrame(transactions_data)
        print(transactions_df.to_string(index=False))
        if all_transactions.count() > 10:
            print(f"... and {all_transactions.count() - 10} more")
    
    # Get import records
    import_records = DocumentImport.objects.filter(user=test_user).order_by('-created')
    print(f"\n📋 IMPORT RECORDS: {import_records.count()} total")
    
    if import_records.exists():
        imports_data = []
        for record in import_records:
            imports_data.append({
                'id': record.id,
                'type': record.get_document_type_display(),
                'filename': record.original_filename,
                'status': record.status,
                'count': record.imported_transactions_count,
                'created': record.created.strftime('%Y-%m-%d %H:%M')
            })
        
        imports_df = pd.DataFrame(imports_data)
        print(imports_df.to_string(index=False))

# Analyze the imported data
print("ANALYZING IMPORTED DATA")
print("="*30)
analyze_imported_data()

## 7. API Testing (Optional)

Testing the REST API endpoints for document upload and management.
Note: This requires the Django development server to be running.

In [ ]:
# API Testing (requires server to be running)
def test_api_endpoints():
    """Test the REST API endpoints"""
    
    base_url = "http://localhost:8000/api/data-sources"
    
    # Test getting supported document types
    try:
        response = requests.get(f"{base_url}/import/types/", timeout=5)
        if response.status_code == 200:
            types_data = response.json()
            print("✅ Supported document types endpoint working:")
            for doc_type in types_data:
                print(f"   - {doc_type['code']}: {doc_type['display']}")
        else:
            print(f"❌ Document types endpoint failed: {response.status_code}")
    except requests.RequestException as e:
        print(f"⚠️  API testing skipped - server not running: {e}")
        return
    
    # Test listing imports (requires authentication)
    print("\n📋 API endpoints are available for testing with proper authentication.")
    print("   To test file upload, use:")
    print(f"   curl -X POST -H 'Authorization: Token YOUR_TOKEN' \\")
    print(f"        -F 'file=@{monthly_report_file}' \\")
    print(f"        -F 'document_type=BANCO_INTER_MONTHLY_REPORT' \\")
    print(f"        {base_url}/import/upload/")

test_api_endpoints()

## 8. Cleanup

Cleaning up temporary files created during testing.

In [ ]:
# Cleanup temporary files
import os

temp_files = [monthly_report_file, brokerage_note_file, extract_file]

for file_path in temp_files:
    try:
        os.unlink(file_path)
        print(f"🗑️  Cleaned up: {file_path}")
    except OSError:
        print(f"⚠️  Could not clean up: {file_path}")

print("\n✅ Testing completed successfully!")

## Summary

This notebook tested the complete Banco Inter document import functionality including:

✅ **Brazilian Number Format Parsing** - Correctly handles R$ 1.234,56 format and negative values in parentheses  
✅ **Date Parsing** - Supports both standard (dd/mm/yyyy) and Portuguese (dd de mês de yyyy) formats  
✅ **CSV Import** - Monthly reports, brokerage notes, and bank extracts  
✅ **PDF Processing** - Full parsing of consolidated reports with position and transaction extraction  
✅ **Asset Management** - Automatic creation of new assets and portfolio integration  
✅ **Data Persistence** - Proper storage of positions, transactions, and import records  

### Key Features Demonstrated:

1. **Four Document Types Supported**:
   - Monthly Investment Reports (CSV)
   - Brokerage Notes (CSV) 
   - Bank Statements (CSV)
   - Consolidated Reports (PDF) - NEW

2. **Robust Data Processing**:
   - Brazilian number format parsing
   - Portuguese date recognition
   - Intelligent asset symbol extraction
   - Flexible column matching

3. **Complete Integration**:
   - Automatic asset creation
   - Portfolio management
   - Transaction tracking
   - Import audit trail

The system is ready for production use with Brazilian financial documents from Banco Inter.